## Imports and paths

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# File paths
PROJECT_DIR = Path.cwd()
INPUT_DIR = (PROJECT_DIR/ "Cleaned Data"/ "003 Column Names update")
OUTPUT_DIR = (PROJECT_DIR/ "Cleaned Data"/ "004 Split")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Split Preparation

In [3]:
# Countries
COUNTRIES = {"AUS": "Australia","BRA": "Brazil","CAN": "Canada","GBR": "United Kingdom","IND": "India","USA": "United States",}


# Split settings
TEST_SIZE = 0.20
RANDOM_STATE = 1948883                                                      # My Student ID


# Core columns
ID_COL = "Record ID"                                                        # ID
DATE_COL = "Survey Date"                                                    # Date
PERIOD_COL = "Mandate Period"                                               # Mandate period


# Two formal target variables
MASK_TARGET_COL = "Face Mask Wearing Binary"                                # Face mask wearing    
PROTECTIVE_TARGET_COL = "Overall Protective Behavior Binary"                # Overall protective behavior


# Outcome-related columns used to control target leakage
MASK_SCORE_COL = "Face Mask Wearing Score"                                  # Face mask wearing
PROTECTIVE_SCORE_COL = "Overall Protective Behavior Score"                  # Overall protective behavior
NON_MASK_SCORE_COL = "Non-Mask Protective Behavior Score"                   # Kept only for mask task
NON_MASK_TARGET_COL = "Non-Mask Protective Behavior Binary"                 # Dropped

## 1 Read

In [4]:
# Read readable encoded datasets
readable_data = {}

for code, country_name in COUNTRIES.items():
    file_path = INPUT_DIR / f"{country_name} (readable).csv"
    readable_data[code] = pd.read_csv(file_path, encoding="utf-8-sig", keep_default_na=False )

## 2 Helper functions

In [5]:
# Check columns exist
def validate_input_columns(df, country_name):
    required_cols =[ ID_COL, DATE_COL, PERIOD_COL, MASK_TARGET_COL, PROTECTIVE_TARGET_COL, MASK_SCORE_COL,PROTECTIVE_SCORE_COL,NON_MASK_SCORE_COL,NON_MASK_TARGET_COL]
    missing_cols = [ col for col in required_cols  if col not in df.columns]
    if len(missing_cols) > 0:
        raise ValueError(f"{country_name} is missing required columns: {missing_cols}")


# Convert mandate period column to integer 0/1
def normalize_period_column(df):
    out = df.copy()                                                                         # Copy
    out[PERIOD_COL] = pd.to_numeric( out[PERIOD_COL], errors="coerce").astype(int)        
    valid_values = set(out[PERIOD_COL].unique().tolist())
    if not valid_values.issubset({0, 1}):
        raise ValueError("Mandate Period must only contain 0 and 1.")
    return out


# Convert binary target column to integer 0/1
def normalize_binary_target(y):
    y_clean = y.copy()                                                                      # Copy
    try:                                                                                    # If target is already numeric 0/1, then .....
        y_num = pd.to_numeric(y_clean, errors="raise").astype(int)
        valid_values = set(y_num.dropna().unique().tolist())
        if valid_values.issubset({0, 1}):
            return y_num.rename("y")
    except Exception:
        pass

    mapping = {"0": 0, "1": 1, "false": 0,"true": 1, "no": 0, "yes": 1}                    # If target is stored as text, then .....
    y_str = y_clean.astype(str).str.strip().str.lower()
    if not set(y_str.unique()).issubset(set(mapping.keys())):
        raise ValueError("Target variable must be binary.")
    return y_str.map(mapping).astype(int).rename("y")



# Convert TRUE/FALSE to 0/1
def normalize_feature_columns(X):
    out = X.copy()                                                                                                      # Copy

    # Convert boolean columns
    bool_cols = out.select_dtypes(include=["bool"]).columns.tolist()                                                    # Change bool to 0/1
    if len(bool_cols) > 0:
        out[bool_cols] = out[bool_cols].astype(int)

    # Convert string TRUE/FALSE columns
    for col in out.columns:                                                                                             # Change String to 0/1
        if out[col].dtype == "object":
            values = set(out[col].dropna().astype(str).str.strip().str.upper().unique().tolist())
            if values.issubset({"TRUE", "FALSE"}):
                out[col] = (out[col].astype(str).str.strip().str.upper().map({"TRUE": 1, "FALSE": 0}).astype(int))
                
    # Check
    for col in out.columns:                                                                                             # Check: whether all predictors is numeric
        out[col] = pd.to_numeric(out[col], errors="raise")
    return out


# Split function
def split_task_data(X, y):
    class_counts = y.value_counts(dropna=False).to_dict()
    use_stratify = ( y.nunique() == 2 and min(class_counts.values()) >= 2)                                                                                     # 80% training set, 20% test set
    X_train, X_test, y_train, y_test = train_test_split( X,  y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y if use_stratify else None )         # RANDOM_STATE = My ID
    return X_train, X_test, y_train, y_test


# Save function
def save_split_files( X_train, X_test, y_train, y_test, task_name, output_dir):

    output_dir.mkdir(parents=True, exist_ok=True)                                                                       # X_train_                            
    X_train.to_csv( output_dir / f"X_train_{task_name}.csv", index=False, encoding="utf-8-sig" )                        # X_test_
    X_test.to_csv( output_dir / f"X_test_{task_name}.csv", index=False, encoding="utf-8-sig" )                          # y_train_    
                                                                                                                        
    pd.DataFrame({"y": y_train}).to_csv( output_dir / f"y_train_{task_name}.csv", index=False, encoding="utf-8-sig")    # y_test_    
    pd.DataFrame({"y": y_test}).to_csv(output_dir / f"y_test_{task_name}.csv", index=False,encoding="utf-8-sig")


# Build and save function
def build_one_task(df_subset,target_col,drop_cols,task_name,output_dir):
    drop_cols = [ col for col in drop_cols if col in df_subset.columns]
    X = df_subset.drop(columns=drop_cols).copy()                                                                         # Predictor matrix
    y = normalize_binary_target(df_subset[target_col])                                                                   # Target vector
    X = normalize_feature_columns(X)
    if X.empty:
        raise ValueError(f"{task_name} produced an empty X matrix.")
    X_train, X_test, y_train, y_test = split_task_data(X, y)
    save_split_files( X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, task_name=task_name, output_dir=output_dir)
    return {"task_name": task_name, "target": target_col, "n_total": len(df_subset),"n_train": len(X_train), "n_test": len(X_test), "n_features": X.shape[1], "positive_rate_total": round(float(y.mean()), 6), "positive_rate_train": round(float(y_train.mean()), 6), "positive_rate_test": round(float(y_test.mean()), 6),}

## 3 Split

In [6]:
split_summary_rows = []                                                             # Save the summary information
PERIOD_LABELS = { 0: "before_mandate", 1: "after_mandate"}                          # 0 is before_mandate, 1 is after_mandate


for code, country_name in COUNTRIES.items():

    df = readable_data[code].copy()                                                 # Retrieve data
    validate_input_columns(df=df,country_name=country_name)                         # Check coulmns
    df = normalize_period_column(df)                                                # Normalize
    safe_country_name = country_name.replace(" ", "_")                              # File name
    country_output_dir = OUTPUT_DIR / safe_country_name
    country_output_dir.mkdir(parents=True, exist_ok=True)

    for period_value, period_label in PERIOD_LABELS.items():
        df_period = df[df[PERIOD_COL] == period_value].copy()                       # Filter data for the current mandate period
        if df_period.shape[0] == 0:
            print(f"Skipped {country_name} - {period_label}: no rows")
            continue


        # Columns removed from both modelling tasks
        common_drop_cols = [ ID_COL,  DATE_COL, PERIOD_COL, MASK_SCORE_COL, MASK_TARGET_COL, PROTECTIVE_SCORE_COL, PROTECTIVE_TARGET_COL, NON_MASK_TARGET_COL]


        # Face mask wearing
        mask_drop_cols = common_drop_cols                                        # For mask wearing: Non-Mask Protective Behavior Score is kept as a predictor
        mask_task_name = f"{safe_country_name}_{period_label}_mask"
        mask_summary = build_one_task( df_subset=df_period, target_col=MASK_TARGET_COL, drop_cols=mask_drop_cols, task_name=mask_task_name, output_dir=country_output_dir )
        mask_summary["country"] = country_name
        mask_summary["mandate_period"] = period_label
        mask_summary["task_type"] = "mask"
        split_summary_rows.append(mask_summary)
        print( f"Saved {mask_task_name}" )


        # Overall protective behavior
        protective_drop_cols = common_drop_cols + [ NON_MASK_SCORE_COL ]          # For overall protective behavior prediction:Non-Mask Protective Behavior Score is removed to avoid target leakage
        protective_task_name = f"{safe_country_name}_{period_label}_protective"
        protective_summary = build_one_task( df_subset=df_period, target_col=PROTECTIVE_TARGET_COL, drop_cols=protective_drop_cols, task_name=protective_task_name, output_dir=country_output_dir )
        protective_summary["country"] = country_name
        protective_summary["mandate_period"] = period_label
        protective_summary["task_type"] = "protective"
        split_summary_rows.append(protective_summary)
        print( f"Saved {protective_task_name}" )

Saved Australia_before_mandate_mask
Saved Australia_before_mandate_protective
Saved Australia_after_mandate_mask
Saved Australia_after_mandate_protective
Saved Brazil_before_mandate_mask
Saved Brazil_before_mandate_protective
Saved Brazil_after_mandate_mask
Saved Brazil_after_mandate_protective
Saved Canada_before_mandate_mask
Saved Canada_before_mandate_protective
Saved Canada_after_mandate_mask
Saved Canada_after_mandate_protective
Saved United_Kingdom_before_mandate_mask
Saved United_Kingdom_before_mandate_protective
Saved United_Kingdom_after_mandate_mask
Saved United_Kingdom_after_mandate_protective
Saved India_before_mandate_mask
Saved India_before_mandate_protective
Saved India_after_mandate_mask
Saved India_after_mandate_protective
Saved United_States_before_mandate_mask
Saved United_States_before_mandate_protective
Saved United_States_after_mandate_mask
Saved United_States_after_mandate_protective
